
# EEG Dataset Consistency Analysis (Notebook Version)

这个 notebook 用于检测 4 个 `.npz` EEG 数据集的一致性，并生成统计结果与可视化图。

## 目标
- 判断 **HBN + MODMA + PRED+CT** 是否可以同时用于预训练。
- 将 **TDBRAIN** 作为下游数据集进行对照。
- 生成统计表和图像：
  - 数据集统计信息
  - 预处理元数据对比
  - FC 有效性验证
  - 通道重叠表
  - 频带功率统计
  - PSD Jensen-Shannon divergence
  - 可视化：RMS、PSD、FFT、频带功率、脑地形图、功能连接图

## 图像要求
- **PSD 保持原来的单图对比形式**。
- **其它图全部改为 4 个子图组成的完整对比图（2×2）**。


In [17]:

from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Dict, List, Tuple, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import welch
from scipy.spatial.distance import jensenshannon

try:
    import mne
except ImportError as e:
    raise SystemExit(
        "MNE is required for topomaps. Install with: mamba install -c conda-forge mne"
    ) from e

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 220


In [18]:

# ----------------------------- Configuration ----------------------------- #

PRETRAIN_DATASETS = ("HBN", "MODMA", "PRED+CT")
DOWNSTREAM_DATASET = "TDBRAIN"

BANDS = {
    "delta": (0.5, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta": (13.0, 30.0),
    "gamma": (30.0, 45.0),
}

REQUIRED_KEYS = ("data", "channel_names", "sfreq", "data_unit")
EXPECTED_SFREQ = 250.0
EXPECTED_WINDOW_SAMPLES = 250
EXPECTED_PRETRAIN_CHANNELS = 64
FC_SYMMETRY_TOL = 1e-5
FC_RECOMPUTE_TOL = 1e-4


In [19]:

# ------------------------------- Utilities ------------------------------- #

def scalar(x: np.ndarray | Any) -> Any:
    if isinstance(x, np.ndarray) and x.ndim == 0:
        return x.item()
    return x


def safe_json_load(value: Any) -> Dict[str, Any]:
    try:
        return json.loads(str(scalar(value)))
    except Exception:
        return {}


def canonical_text(x: Any) -> str:
    if isinstance(x, (dict, list, tuple)):
        return json.dumps(x, sort_keys=True, ensure_ascii=False)
    return str(x)


def ensure_3d_eeg(x: np.ndarray, name: str) -> np.ndarray:
    if x.ndim != 3:
        raise ValueError(f"{name}: data must be 3-D (N,C,T), got {x.shape}")
    return x


def load_npz(path: Path, dataset_name: str) -> Dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(path)

    with np.load(path, allow_pickle=True) as z:
        missing = [k for k in REQUIRED_KEYS if k not in z.files]
        if missing:
            raise KeyError(f"{dataset_name}: missing keys: {missing}")

        data = ensure_3d_eeg(np.asarray(z["data"], dtype=np.float32), dataset_name)
        channel_names = np.asarray(z["channel_names"]).astype(str).tolist()
        sfreq = float(scalar(z["sfreq"]))
        data_unit = str(scalar(z["data_unit"]))
        metadata = safe_json_load(z["metadata_json"]) if "metadata_json" in z.files else {}
        labels = np.asarray(z["labels"]).astype(str) if "labels" in z.files else np.array([], dtype=str)
        fc = np.asarray(z["fc"], dtype=np.float32) if "fc" in z.files else None

    if len(channel_names) != data.shape[1]:
        raise ValueError(
            f"{dataset_name}: channel_names has {len(channel_names)} items but data has C={data.shape[1]}"
        )

    return {
        "name": dataset_name,
        "path": path,
        "data": data,
        "fc": fc,
        "channel_names": channel_names,
        "sfreq": sfreq,
        "data_unit": data_unit,
        "labels": labels,
        "metadata": metadata,
    }


def compute_mean_psd(data: np.ndarray, sfreq: float) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    nperseg = min(data.shape[-1], int(round(sfreq)))
    noverlap = nperseg // 2
    f, pxx = welch(
        data.astype(np.float64),
        fs=sfreq,
        nperseg=nperseg,
        noverlap=noverlap,
        detrend="constant",
        axis=-1,
        scaling="density",
    )
    mean_psd = np.mean(pxx, axis=(0, 1))
    channel_psd = np.mean(pxx, axis=0)
    return f, mean_psd, channel_psd


def compute_mean_fft(data: np.ndarray, sfreq: float) -> Tuple[np.ndarray, np.ndarray]:
    x = data.astype(np.float64)
    x = x - x.mean(axis=-1, keepdims=True)
    spec = np.abs(np.fft.rfft(x, axis=-1)) * (2.0 / x.shape[-1])
    freq = np.fft.rfftfreq(x.shape[-1], d=1.0 / sfreq)
    return freq, np.mean(spec, axis=(0, 1))


def band_mask(freqs: np.ndarray, lo: float, hi: float, is_last: bool = False) -> np.ndarray:
    if is_last:
        return (freqs >= lo) & (freqs <= hi)
    return (freqs >= lo) & (freqs < hi)


def relative_band_power(freqs: np.ndarray, psd: np.ndarray) -> Dict[str, float]:
    total_mask = (freqs >= 0.5) & (freqs <= 45.0)
    total = float(np.sum(psd[total_mask]))
    out = {}
    for i, (band, (lo, hi)) in enumerate(BANDS.items()):
        mask = band_mask(freqs, lo, hi, is_last=(i == len(BANDS) - 1))
        value = float(np.sum(psd[mask]))
        out[band] = 100.0 * value / total if total > 0 else np.nan
    return out


def channel_band_power(freqs: np.ndarray, channel_psd: np.ndarray, band: str = "alpha") -> np.ndarray:
    lo, hi = BANDS[band]
    band_m = band_mask(freqs, lo, hi, is_last=(band == list(BANDS)[-1]))
    total_m = (freqs >= 0.5) & (freqs <= 45.0)
    num = np.sum(channel_psd[:, band_m], axis=1)
    den = np.sum(channel_psd[:, total_m], axis=1)
    return 100.0 * np.divide(num, den, out=np.full_like(num, np.nan), where=den > 0)


def compute_fc_from_window(window: np.ndarray) -> np.ndarray:
    c = np.corrcoef(window)
    c = np.nan_to_num(c, nan=0.0, posinf=0.0, neginf=0.0)
    np.fill_diagonal(c, 0.0)
    return c


def validate_fc(ds: Dict[str, Any], max_windows: int = 20) -> Dict[str, Any]:
    data = ds["data"]
    fc = ds["fc"]
    c = data.shape[1]

    if fc is None:
        return {
            "dataset": ds["name"],
            "has_fc": False,
            "shape_ok": False,
            "symmetric_max_error": np.nan,
            "diagonal_abs_max": np.nan,
            "recompute_max_abs_error": np.nan,
            "fc_min": np.nan,
            "fc_max": np.nan,
            "mean_abs_offdiag": np.nan,
        }

    shape_ok = fc.shape == (data.shape[0], c, c)
    sym_err = float(np.max(np.abs(fc - np.swapaxes(fc, 1, 2)))) if shape_ok else np.nan
    diag = np.diagonal(fc, axis1=1, axis2=2) if shape_ok else np.array([np.nan])
    diag_err = float(np.nanmax(np.abs(diag))) if shape_ok else np.nan

    rec_errs = []
    if shape_ok:
        for i in range(min(max_windows, data.shape[0])):
            rec = compute_fc_from_window(data[i])
            rec_errs.append(float(np.max(np.abs(rec - fc[i]))))
    rec_err = max(rec_errs) if rec_errs else np.nan

    offdiag_mask = ~np.eye(c, dtype=bool)
    mean_abs = float(np.mean(np.abs(fc[:, offdiag_mask]))) if shape_ok else np.nan

    return {
        "dataset": ds["name"],
        "has_fc": True,
        "shape_ok": bool(shape_ok),
        "symmetric_max_error": sym_err,
        "diagonal_abs_max": diag_err,
        "recompute_max_abs_error": rec_err,
        "fc_min": float(np.min(fc)) if shape_ok else np.nan,
        "fc_max": float(np.max(fc)) if shape_ok else np.nan,
        "mean_abs_offdiag": mean_abs,
    }


def dataset_summary(ds: Dict[str, Any]) -> Dict[str, Any]:
    x = ds["data"].astype(np.float64)
    window_rms = np.sqrt(np.mean(x ** 2, axis=(1, 2)))
    window_max_p2p = np.ptp(x, axis=-1).max(axis=1)
    labels = sorted(set(ds["labels"].tolist())) if ds["labels"].size else []

    return {
        "dataset": ds["name"],
        "file": str(ds["path"]),
        "shape": str(tuple(x.shape)),
        "n_windows": x.shape[0],
        "n_channels": x.shape[1],
        "n_samples": x.shape[2],
        "sfreq_hz": ds["sfreq"],
        "window_sec": x.shape[2] / ds["sfreq"],
        "unit": ds["data_unit"],
        "dtype": str(ds["data"].dtype),
        "labels": " | ".join(labels),
        "nan_count": int(np.isnan(x).sum()),
        "inf_count": int(np.isinf(x).sum()),
        "mean_uV": float(np.mean(x)),
        "std_uV": float(np.std(x)),
        "median_abs_uV": float(np.median(np.abs(x))),
        "max_abs_uV": float(np.max(np.abs(x))),
        "median_window_rms_uV": float(np.median(window_rms)),
        "p95_window_rms_uV": float(np.percentile(window_rms, 95)),
        "median_window_max_channel_p2p_uV": float(np.median(window_max_p2p)),
    }


def preprocessing_row(ds: Dict[str, Any]) -> Dict[str, Any]:
    m = ds["metadata"]
    keys = [
        "dataset_name",
        "target_sfreq",
        "window_seconds",
        "bandpass_hz",
        "filter",
        "notch_hz",
        "reference",
        "ica_method",
        "baseline_correction",
        "bad_channel_method",
        "interpolation_method",
        "functional_connectivity",
        "windowing",
    ]
    row = {"dataset": ds["name"]}
    for k in keys:
        row[k] = canonical_text(m.get(k, ""))
    return row


def mean_fc(ds: Dict[str, Any]) -> np.ndarray:
    if ds["fc"] is not None:
        return np.mean(ds["fc"].astype(np.float64), axis=0)
    mats = [compute_fc_from_window(w) for w in ds["data"]]
    return np.mean(mats, axis=0)


def shared_channels(datasets: Dict[str, Dict[str, Any]]) -> List[str]:
    base = datasets[DOWNSTREAM_DATASET]["channel_names"]
    common = set(base)
    for ds in datasets.values():
        common &= set(ds["channel_names"])
    return [c for c in base if c in common]


def reindex_matrix(mat: np.ndarray, source_ch: List[str], target_ch: List[str]) -> np.ndarray:
    idx = [source_ch.index(c) for c in target_ch]
    return mat[np.ix_(idx, idx)]


def channel_overlap_table(datasets: Dict[str, Dict[str, Any]]) -> pd.DataFrame:
    all_names = []
    for ds in datasets.values():
        for ch in ds["channel_names"]:
            if ch not in all_names:
                all_names.append(ch)
    rows = []
    for ch in all_names:
        row = {"channel": ch}
        for name, ds in datasets.items():
            row[name] = ch in ds["channel_names"]
        rows.append(row)
    return pd.DataFrame(rows)


def build_common_montage(
    datasets: Dict[str, Dict[str, Any]],
    channel_names: List[str],
) -> mne.channels.DigMontage:
    mapping = None
    for name in PRETRAIN_DATASETS:
        m = datasets[name]["metadata"]
        candidate = m.get("common64_to_hydrocel")
        if not candidate:
            candidate = m.get("source_info", {}).get("common64_to_hydrocel")
        if candidate:
            mapping = candidate
            break

    if mapping:
        hydro = mne.channels.make_standard_montage("GSN-HydroCel-129")
        pos_src = hydro.get_positions()["ch_pos"]
        ch_pos = {}
        for ch in channel_names:
            src = mapping.get(ch)
            if src in pos_src:
                ch_pos[ch] = pos_src[src]
        if len(ch_pos) >= max(4, int(0.8 * len(channel_names))):
            return mne.channels.make_dig_montage(ch_pos=ch_pos, coord_frame="head")

    std = mne.channels.make_standard_montage("standard_1005")
    pos_src = std.get_positions()["ch_pos"]
    ch_pos = {ch: pos_src[ch] for ch in channel_names if ch in pos_src}
    if len(ch_pos) < 4:
        raise RuntimeError("Not enough electrode positions to make a topomap")
    return mne.channels.make_dig_montage(ch_pos=ch_pos, coord_frame="head")


In [20]:

# ----------------------------- Compatibility ----------------------------- #

def check_pretraining_compatibility(
    datasets: Dict[str, Dict[str, Any]],
    summaries: pd.DataFrame,
    fc_df: pd.DataFrame,
) -> Tuple[List[str], bool]:
    msgs: List[str] = []
    pre = [datasets[n] for n in PRETRAIN_DATASETS]

    def add(level: str, text: str):
        msgs.append(f"[{level}] {text}")

    structural_pass = True

    for ds in pre:
        x = ds["data"]
        if x.shape[1] != EXPECTED_PRETRAIN_CHANNELS:
            add("FAIL", f"{ds['name']}: C={x.shape[1]}, expected {EXPECTED_PRETRAIN_CHANNELS}.")
            structural_pass = False
        if x.shape[2] != EXPECTED_WINDOW_SAMPLES:
            add("FAIL", f"{ds['name']}: T={x.shape[2]}, expected {EXPECTED_WINDOW_SAMPLES}.")
            structural_pass = False
        if not math.isclose(ds["sfreq"], EXPECTED_SFREQ, rel_tol=0, abs_tol=1e-6):
            add("FAIL", f"{ds['name']}: sfreq={ds['sfreq']}, expected {EXPECTED_SFREQ} Hz.")
            structural_pass = False
        if ds["data_unit"].lower() not in {"uv", "µv", "μv"}:
            add("FAIL", f"{ds['name']}: unit={ds['data_unit']!r}; expected uV.")
            structural_pass = False
        if not np.isfinite(ds["data"]).all():
            add("FAIL", f"{ds['name']}: EEG contains NaN/Inf.")
            structural_pass = False

    ref = pre[0]["channel_names"]
    for ds in pre[1:]:
        if ds["channel_names"] != ref:
            if set(ds["channel_names"]) == set(ref):
                add("FAIL", f"{ds['name']}: same channel set but channel ORDER differs from HBN.")
            else:
                add("FAIL", f"{ds['name']}: channel set differs from HBN.")
            structural_pass = False
    if all(ds["channel_names"] == ref for ds in pre):
        add("PASS", "HBN, MODMA and PRED+CT have identical 64-channel names and order.")

    for ds in pre:
        labels = set(ds["labels"].tolist()) if ds["labels"].size else set()
        if labels and labels != {"Eyes Closed"}:
            add("WARN", f"{ds['name']}: labels are {sorted(labels)}, not EC-only.")
    if all(set(ds["labels"].tolist()) == {"Eyes Closed"} for ds in pre if ds["labels"].size):
        add("PASS", "The three candidate pretraining files are Eyes Closed only.")

    for _, r in fc_df.iterrows():
        if r["dataset"] not in PRETRAIN_DATASETS:
            continue
        if not bool(r["has_fc"]) or not bool(r["shape_ok"]):
            add("FAIL", f"{r['dataset']}: FC missing or shape mismatch.")
            structural_pass = False
        else:
            if r["symmetric_max_error"] > FC_SYMMETRY_TOL:
                add("FAIL", f"{r['dataset']}: FC is not symmetric enough.")
                structural_pass = False
            if r["recompute_max_abs_error"] > FC_RECOMPUTE_TOL:
                add("FAIL", f"{r['dataset']}: stored FC does not match recomputed Pearson FC.")
                structural_pass = False
    if structural_pass:
        add("PASS", "Stored FC tensors satisfy shape/symmetry/recomputation checks for pretraining.")

    td = datasets[DOWNSTREAM_DATASET]
    missing = [c for c in td["channel_names"] if c not in ref]
    if missing:
        add("WARN", f"TDBRAIN has channels absent from the 64-channel pretraining space: {missing}")
    else:
        add("PASS", f"All {len(td['channel_names'])} TDBRAIN EEG channels are a subset of the pretraining 64 channels.")

    compare_fields = ["bandpass_hz", "reference", "baseline_correction", "bad_channel_method", "interpolation_method"]
    for field in compare_fields:
        vals = [canonical_text(ds["metadata"].get(field, "")) for ds in pre]
        if len(set(vals)) == 1:
            add("PASS", f"Preprocessing field '{field}' is aligned: {vals[0]}")
        else:
            add("WARN", f"Preprocessing field '{field}' differs: " + ", ".join(f"{d['name']}={v}" for d, v in zip(pre, vals)))

    ica_vals = [canonical_text(ds["metadata"].get("ica_method", "")) for ds in pre]
    if len(set(ica_vals)) > 1:
        add("WARN", "ICA policy differs across pretraining datasets: " + ", ".join(f"{d['name']}={v}" for d, v in zip(pre, ica_vals)))

    notch_vals = [canonical_text(ds["metadata"].get("notch_hz", "")) for ds in pre]
    if len(set(notch_vals)) > 1:
        add("INFO", "Notch frequencies differ (50/60 Hz). This is normally acceptable when matched to mains frequency, especially because the final bandpass is 0.5-45 Hz.")

    sub = summaries[summaries["dataset"].isin(PRETRAIN_DATASETS)].copy()
    rms = sub.set_index("dataset")["median_window_rms_uV"]
    ratio = float(rms.max() / max(rms.min(), 1e-12))
    if ratio >= 2.0:
        add("WARN", f"Median window RMS differs by {ratio:.2f}x across pretraining datasets. Consider input normalization / dataset-balanced sampling to prevent dataset-identity shortcuts.")
    else:
        add("PASS", f"Median window RMS scale ratio is {ratio:.2f}x across pretraining datasets.")

    if structural_pass:
        add("RESULT", "STRUCTURAL COMPATIBILITY = PASS. HBN + MODMA + PRED+CT can be concatenated for pretraining under the current tensor contract.")
        add("RESULT", "TDBRAIN should remain downstream-only; map/freeze the encoder using its 26-channel subset rather than padding it into the 64-channel pretraining pool unless your architecture explicitly supports channel masking.")
    else:
        add("RESULT", "STRUCTURAL COMPATIBILITY = FAIL. Fix FAIL items before mixed pretraining.")

    return msgs, structural_pass


In [21]:

# ------------------------------- Plotting -------------------------------- #

def savefig(fig, path: Path):
    # Layout is controlled explicitly in each plotting function.
    # Avoid global tight_layout(), which can conflict with shared colorbars
    # and make 2x2 panels look too far apart or cause label overlap.

    # Transparent figure + all axes (including colorbar axes)
    fig.patch.set_alpha(0)
    for ax in fig.axes:
        ax.patch.set_alpha(0)

    fig.savefig(
        path,
        dpi=220,
        bbox_inches="tight",
        transparent=True,
        facecolor="none",
        edgecolor="none",
    )
    plt.close(fig)


def plot_rms_distribution_comparison(datasets: Dict[str, Dict[str, Any]], out: Path):
    fig, axes = plt.subplots(2, 2, figsize=(9.4, 6.8))
    axes = axes.flatten()
    for ax, (name, ds) in zip(axes, datasets.items()):
        x = ds["data"].astype(np.float64)
        rms = np.sqrt(np.mean(x ** 2, axis=(1, 2)))
        ax.hist(rms, bins=25)
        ax.axvline(np.median(rms), linestyle="--", linewidth=1.2, label=f"median={np.median(rms):.2f}")
        ax.set_title(name)
        ax.set_xlabel("Window RMS (uV)")
        ax.set_ylabel("Count")
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)
    fig.suptitle("Window RMS distribution comparison", fontsize=14, y=0.975)
    fig.subplots_adjust(left=0.10, right=0.98, bottom=0.10, top=0.90,
                        wspace=0.22, hspace=0.34)
    savefig(fig, out / "01_window_rms_distribution_4panel.png")


def plot_mean_psd(psd_data: Dict[str, Tuple[np.ndarray, np.ndarray, np.ndarray]], out: Path):
    fig, ax = plt.subplots(figsize=(9, 5.5))
    for name, (f, mean_psd, _) in psd_data.items():
        m = (f >= 0.5) & (f <= 45)
        ax.semilogy(f[m], mean_psd[m], label=name)
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("PSD (uV^2/Hz)")
    ax.set_title("Mean Welch PSD")
    ax.legend()
    ax.grid(alpha=0.25)
    savefig(fig, out / "02_mean_welch_psd.png")


def plot_mean_fft_comparison(datasets: Dict[str, Dict[str, Any]], out: Path):
    fig, axes = plt.subplots(2, 2, figsize=(9.4, 6.8), sharex=True, sharey=True)
    axes = axes.flatten()
    for ax, (name, ds) in zip(axes, datasets.items()):
        f, amp = compute_mean_fft(ds["data"], ds["sfreq"])
        m = (f >= 0.5) & (f <= 45)
        ax.plot(f[m], amp[m])
        ax.set_title(name)
        ax.set_xlabel("Frequency (Hz)")
        ax.set_ylabel("Mean FFT amplitude (uV)")
        ax.grid(alpha=0.25)
    fig.suptitle("Mean amplitude spectrum comparison", fontsize=14, y=0.975)
    fig.subplots_adjust(left=0.11, right=0.98, bottom=0.10, top=0.90,
                        wspace=0.16, hspace=0.30)
    savefig(fig, out / "03_mean_fft_spectrum_4panel.png")


def plot_band_power_comparison(band_df: pd.DataFrame, out: Path):
    fig, axes = plt.subplots(2, 2, figsize=(9.4, 6.8), sharey=True)
    axes = axes.flatten()
    band_order = list(BANDS.keys())
    for ax, dataset in zip(axes, band_df["dataset"].unique()):
        sub = band_df[band_df["dataset"] == dataset].set_index("band").reindex(band_order)
        ax.bar(sub.index, sub["relative_power_pct"].values)
        ax.set_title(dataset)
        ax.set_xlabel("Band")
        ax.set_ylabel("Relative power (%)")
        ax.grid(axis="y", alpha=0.25)
        ax.tick_params(axis='x', rotation=0)
    fig.suptitle("Relative spectral band power comparison", fontsize=14, y=0.975)
    fig.subplots_adjust(left=0.10, right=0.98, bottom=0.10, top=0.90,
                        wspace=0.18, hspace=0.30)
    savefig(fig, out / "04_relative_band_power_4panel.png")


def prepare_topomap_values(ds: Dict[str, Any], values: np.ndarray, montage: mne.channels.DigMontage):
    ch_names = ds["channel_names"]
    available = set(montage.ch_names)
    keep = [i for i, ch in enumerate(ch_names) if ch in available and np.isfinite(values[i])]
    keep_names = [ch_names[i] for i in keep]
    keep_vals = values[keep]
    if len(keep) < 4:
        return None, None
    info = mne.create_info(keep_names, sfreq=ds["sfreq"], ch_types="eeg")
    info.set_montage(montage, on_missing="ignore")
    return info, keep_vals


def plot_topomap_comparison(alpha_map: Dict[str, np.ndarray], datasets: Dict[str, Dict[str, Any]], montage: mne.channels.DigMontage, out: Path):
    prepared = {}
    all_vals = []
    for name, ds in datasets.items():
        info, vals = prepare_topomap_values(ds, alpha_map[name], montage)
        if info is not None:
            prepared[name] = (info, vals)
            all_vals.append(vals)
    if not all_vals:
        return

    global_min = min(np.nanmin(v) for v in all_vals)
    global_max = max(np.nanmax(v) for v in all_vals)

    fig, axes = plt.subplots(2, 2, figsize=(8.8, 7.0))
    axes = axes.flatten()
    im = None
    for ax, (name, ds) in zip(axes, datasets.items()):
        if name not in prepared:
            ax.set_title(f"{name} (not enough channels)")
            ax.axis("off")
            continue
        info, vals = prepared[name]
        im, _ = mne.viz.plot_topomap(
            vals,
            info,
            axes=ax,
            show=False,
            contours=6,
            sensors=True,
            vlim=(global_min, global_max),
        )
        ax.set_title(name)
    if im is not None:
        cbar = fig.colorbar(
            im, ax=axes.tolist(), shrink=0.82, fraction=0.035, pad=0.035
        )
        cbar.set_label("Relative alpha power (%)", labelpad=8)
    fig.suptitle("Alpha relative-power topomap comparison (8-13 Hz)", fontsize=14, y=0.975)
    fig.subplots_adjust(left=0.07, right=0.88, bottom=0.06, top=0.90,
                        wspace=0.10, hspace=0.20)
    savefig(fig, out / "05_alpha_topomap_comparison.png")


def plot_fc_matrix_comparison(
    matrices: Dict[str, np.ndarray],
    channel_orders: Dict[str, List[str]],
    out_file: Path,
    title: str,
):
    vmax = max(max(np.max(np.abs(mat)), 1e-6) for mat in matrices.values())
    fig, axes = plt.subplots(2, 2, figsize=(10.5, 8.4))
    axes = axes.flatten()
    im = None
    for ax, (name, mat) in zip(axes, matrices.items()):
        ch_names = channel_orders[name]
        im = ax.imshow(mat, vmin=-vmax, vmax=vmax, cmap="RdBu_r", interpolation="nearest")
        n = len(ch_names)
        ticks = np.arange(n) if n <= 32 else np.arange(0, n, 4)
        ax.set_xticks(ticks)
        ax.set_xticklabels([ch_names[i] for i in ticks], rotation=90, fontsize=6)
        ax.set_yticks(ticks)
        ax.set_yticklabels([ch_names[i] for i in ticks], fontsize=6)
        ax.set_title(name)
        ax.set_xlabel("Channel", labelpad=7)
        ax.set_ylabel("Channel", labelpad=7)
    if im is not None:
        cbar = fig.colorbar(
            im, ax=axes.tolist(), shrink=0.82, fraction=0.030, pad=0.035
        )
        cbar.set_label("Pearson r", labelpad=8)
    fig.suptitle(title, fontsize=14, y=0.98)
    # Compact panel spacing while keeping rotated channel labels clear.
    fig.subplots_adjust(left=0.08, right=0.88, bottom=0.10, top=0.91,
                        wspace=0.16, hspace=0.28)
    savefig(fig, out_file)


In [22]:
# ------------------------------- File paths ------------------------------ #

# Windows 本地运行：通常只需要修改这一行。
# 程序会先查找 notebook 当前目录，然后递归搜索 DATA_ROOT。
DATA_ROOT = Path(r"E:\Workspace\dataset\preprocessed")

file_names = {
    "HBN": "sub-NDARCA153NKE_task-RestingState_eeg_EC.npz",
    "MODMA": "02010036_EC.npz",
    "PRED+CT": "sub-001_task-Rest_run-01_eeg_EC.npz",
    "TDBRAIN": "sub-87958057_ses-1_task-restEC_eeg_EC.npz",
}

def resolve_npz(filename: str) -> Path:
    # 1) 与 notebook / 当前工作目录放在一起时，直接使用
    local = Path.cwd() / filename
    if local.exists():
        return local

    # 2) 在统一数据根目录下递归搜索（适配 HBN/MODMA/PREDCT/TDBRAIN 子目录）
    if DATA_ROOT.exists():
        matches = list(DATA_ROOT.rglob(filename))
        if matches:
            return matches[0]

    raise FileNotFoundError(
        f"Cannot find {filename}\n"
        f"Current working directory: {Path.cwd()}\n"
        f"DATA_ROOT: {DATA_ROOT}\n"
        "Please put the .npz beside the notebook or change DATA_ROOT above."
    )

file_map = {name: resolve_npz(filename) for name, filename in file_names.items()}

# 输出到当前工作目录，避免使用 Linux 的 /mnt/data 路径
out = Path.cwd() / "eeg_consistency_report_notebook"
fig_out = out / "figures"
out.mkdir(parents=True, exist_ok=True)
fig_out.mkdir(parents=True, exist_ok=True)

file_map


{'HBN': WindowsPath('E:/Workspace/dataset/preprocessed/HBN/sub-NDARCA153NKE_task-RestingState_eeg_EC.npz'),
 'MODMA': WindowsPath('E:/Workspace/dataset/preprocessed/MODMA/02010036_EC.npz'),
 'PRED+CT': WindowsPath('E:/Workspace/dataset/preprocessed/PRED_CT/sub-001_task-Rest_run-01_eeg_EC.npz'),
 'TDBRAIN': WindowsPath('E:/Workspace/dataset/preprocessed/TDBRAIN/sub-87958057_ses-1_task-restEC_eeg_EC.npz')}

In [23]:

# ----------------------------- Load datasets ----------------------------- #

datasets = {name: load_npz(path, name) for name, path in file_map.items()}
list(datasets.keys())


['HBN', 'MODMA', 'PRED+CT', 'TDBRAIN']

In [24]:

# ----------------------------- Basic summaries --------------------------- #

summary_df = pd.DataFrame([dataset_summary(ds) for ds in datasets.values()])
summary_df.to_csv(out / "dataset_summary.csv", index=False, encoding="utf-8-sig")
summary_df


,dataset,file,shape,n_windows,n_channels,n_samples,sfreq_hz,window_sec,unit,dtype,labels,nan_count,inf_count,mean_uV,std_uV,median_abs_uV,max_abs_uV,median_window_rms_uV,p95_window_rms_uV,median_window_max_channel_p2p_uV
0,HBN,E:\Workspace\dataset\preprocessed\HBN\sub-NDAR...,"(173, 64, 250)",173,64,250,250.0,1.0,uV,float32,Eyes Closed,0,0,-2.361114e-11,8.088001,2.801615,199.140488,6.630890,12.691853,123.366726
1,MODMA,E:\Workspace\dataset\preprocessed\MODMA\020100...,"(296, 64, 250)",296,64,250,250.0,1.0,uV,float32,Eyes Closed,0,0,1.869715e-12,7.342684,4.089421,188.561325,7.086774,10.466549,65.079727
2,PRED+CT,E:\Workspace\dataset\preprocessed\PRED_CT\sub-...,"(201, 64, 250)",201,64,250,250.0,1.0,uV,float32,Eyes Closed,0,0,9.442181e-11,4.725290,2.911644,104.638779,4.628469,5.746195,40.235723
3,TDBRAIN,E:\Workspace\dataset\preprocessed\TDBRAIN\sub-...,"(119, 26, 250)",119,26,250,250.0,1.0,uV,float32,Eyes Closed,0,0,-6.577472e-11,6.876975,3.980267,71.941498,6.599290,9.069964,58.010733


In [25]:

prep_df = pd.DataFrame([preprocessing_row(ds) for ds in datasets.values()])
prep_df.to_csv(out / "preprocessing_metadata.csv", index=False, encoding="utf-8-sig")
prep_df


,dataset,dataset_name,target_sfreq,window_seconds,bandpass_hz,filter,notch_hz,reference,ica_method,baseline_correction,bad_channel_method,interpolation_method,functional_connectivity,windowing
0,HBN,HBN,250.0,1.0,"[0.5, 45.0]",4th-order Butterworth IIR,60.0,common average reference,FastICA with frontal EEG proxy scoring,whole-window mean,"robust z-score of log(SD), |z| > 6.0",3-nearest-electrode inverse-distance weighted ...,Pearson correlation computed independently for...,event-aware HBN resting-state segmentation; co...
1,MODMA,MODMA,250.0,1.0,"[0.5, 45.0]",4th-order Butterworth IIR,50.0,common average reference,FastICA with frontal EEG proxy scoring,whole-window mean,"robust z-score of log(SD), |z| > 6.0",3-nearest-electrode inverse-distance weighted ...,Pearson correlation computed independently for...,full eyes-closed resting recording; complete n...
2,PRED+CT,PRED+CT,250.0,1.0,"[0.5, 45.0]",4th-order Butterworth IIR,60.0,common average reference,FastICA,whole-window mean,"robust z-score of log(SD), |z| > 6.0",3-nearest-electrode inverse-distance weighted ...,Pearson correlation computed independently for...,event-aware; windows never cross event-block b...
3,TDBRAIN,TDBRAIN,250.0,1.0,"[0.5, 45.0]",4th-order Butterworth IIR,50.0,common average reference,FastICA,whole-window mean,"robust z-score of log(SD), |z| > 6.0",3-nearest-electrode inverse-distance weighted ...,Pearson correlation computed independently for...,full task-restEC recording; complete non-overl...


In [26]:

fc_df = pd.DataFrame([validate_fc(ds) for ds in datasets.values()])
fc_df.to_csv(out / "fc_validation.csv", index=False, encoding="utf-8-sig")
fc_df


,dataset,has_fc,shape_ok,symmetric_max_error,diagonal_abs_max,recompute_max_abs_error,fc_min,fc_max,mean_abs_offdiag
0,HBN,True,True,0.0,0.0,3.811766e-07,-0.992113,0.999381,0.474041
1,MODMA,True,True,0.0,0.0,2.975395e-07,-0.978875,0.996737,0.533682
2,PRED+CT,True,True,0.0,0.0,3.269680e-07,-0.934513,0.991541,0.399311
3,TDBRAIN,True,True,0.0,0.0,2.859051e-07,-0.985735,0.992742,0.436666


In [27]:

overlap_df = channel_overlap_table(datasets)
overlap_df.to_csv(out / "channel_overlap.csv", index=False, encoding="utf-8-sig")
overlap_df.head()


,channel,HBN,MODMA,PRED+CT,TDBRAIN
0,Fp1,True,True,True,True
1,Fpz,True,True,True,False
2,Fp2,True,True,True,True
3,AF3,True,True,True,False
4,AF4,True,True,True,False


In [28]:

# --------------------------- PSD and band power -------------------------- #

psd_data = {}
band_rows = []
for name, ds in datasets.items():
    f, mpsd, cpsd = compute_mean_psd(ds["data"], ds["sfreq"])
    psd_data[name] = (f, mpsd, cpsd)
    bp = relative_band_power(f, mpsd)
    for band, value in bp.items():
        band_rows.append({"dataset": name, "band": band, "relative_power_pct": value})

band_df = pd.DataFrame(band_rows)
band_df.to_csv(out / "band_power_relative_pct.csv", index=False, encoding="utf-8-sig")
band_df


,dataset,band,relative_power_pct
0,HBN,delta,79.330858
1,HBN,theta,12.367183
2,HBN,alpha,5.546321
3,HBN,beta,2.301022
4,HBN,gamma,0.454617
5,MODMA,delta,19.977511
6,MODMA,theta,6.680164
7,MODMA,alpha,62.095201
8,MODMA,beta,8.853885
9,MODMA,gamma,2.393239


In [29]:

# ------------------------ PSD JS divergence matrix ----------------------- #

names = list(datasets.keys())
js = pd.DataFrame(index=names, columns=names, dtype=float)
for a in names:
    fa, pa, _ = psd_data[a]
    mask_a = (fa >= 0.5) & (fa <= 45.0)
    va = pa[mask_a]
    va = va / np.sum(va)
    for b in names:
        fb, pb, _ = psd_data[b]
        mask_b = (fb >= 0.5) & (fb <= 45.0)
        vb = pb[mask_b]
        vb = vb / np.sum(vb)
        if len(va) != len(vb) or not np.allclose(fa[mask_a], fb[mask_b]):
            js.loc[a, b] = np.nan
        else:
            js.loc[a, b] = float(jensenshannon(va, vb, base=2.0) ** 2)

js.to_csv(out / "psd_js_divergence.csv", encoding="utf-8-sig")
js


,HBN,MODMA,PRED+CT,TDBRAIN
HBN,0.000000,0.375552,0.200992,0.260636
MODMA,0.375552,0.000000,0.177295,0.037490
PRED+CT,0.200992,0.177295,0.000000,0.078215
TDBRAIN,0.260636,0.037490,0.078215,0.000000


In [30]:
# -------------------------- Compatibility report ------------------------- #

messages, structural_pass = check_pretraining_compatibility(datasets, summary_df, fc_df)
report = [
    "EEG DATASET CONSISTENCY REPORT",
    "=" * 78,
    "Goal: HBN + MODMA + PRED+CT for pretraining; TDBRAIN downstream only.",
    "",
    *messages,
    "",
    "Shared downstream channel order:",
    ", ".join(shared_channels(datasets)),
    "",
    "Interpretation:",
    "- FAIL  : must fix before mixed pretraining.",
    "- WARN  : structurally usable, but domain/preprocessing mismatch may bias SSL.",
    "- INFO  : expected/non-blocking difference.",
    "- PASS  : aligned with the intended tensor/data contract.",
]
(out / "compatibility_report.txt").write_text("\n".join(report), encoding="utf-8")

print("\n".join(report))
print(f"\nStructural pretraining compatibility: {'PASS' if structural_pass else 'FAIL'}")


EEG DATASET CONSISTENCY REPORT
Goal: HBN + MODMA + PRED+CT for pretraining; TDBRAIN downstream only.

[PASS] HBN, MODMA and PRED+CT have identical 64-channel names and order.
[PASS] The three candidate pretraining files are Eyes Closed only.
[PASS] Stored FC tensors satisfy shape/symmetry/recomputation checks for pretraining.
[PASS] All 26 TDBRAIN EEG channels are a subset of the pretraining 64 channels.
[PASS] Preprocessing field 'bandpass_hz' is aligned: [0.5, 45.0]
[PASS] Preprocessing field 'reference' is aligned: common average reference
[PASS] Preprocessing field 'baseline_correction' is aligned: whole-window mean
[PASS] Preprocessing field 'bad_channel_method' is aligned: robust z-score of log(SD), |z| > 6.0
[PASS] Preprocessing field 'interpolation_method' is aligned: 3-nearest-electrode inverse-distance weighted average
[WARN] ICA policy differs across pretraining datasets: HBN=FastICA with frontal EEG proxy scoring, MODMA=FastICA with frontal EEG proxy scoring, PRED+CT=FastIC

In [31]:

# ------------------------------- Figures --------------------------------- #

plot_rms_distribution_comparison(datasets, fig_out)
plot_mean_psd(psd_data, fig_out)  # PSD 保持单图对比
plot_mean_fft_comparison(datasets, fig_out)
plot_band_power_comparison(band_df, fig_out)

montage = build_common_montage(datasets, datasets["HBN"]["channel_names"])
alpha_map = {}
for name, ds in datasets.items():
    f, _, cpsd = psd_data[name]
    alpha_map[name] = channel_band_power(f, cpsd, band="alpha")
plot_topomap_comparison(alpha_map, datasets, montage, fig_out)

mean_fc_cache = {name: mean_fc(ds) for name, ds in datasets.items()}
full_mats = {name: mean_fc_cache[name] for name in datasets}
full_orders = {name: datasets[name]["channel_names"] for name in datasets}
plot_fc_matrix_comparison(
    full_mats,
    full_orders,
    fig_out / "06_mean_fc_full_comparison.png",
    "Mean functional connectivity comparison (full channel space)",
)

common26 = shared_channels(datasets)
common26_mats = {
    name: reindex_matrix(mean_fc_cache[name], datasets[name]["channel_names"], common26)
    for name in datasets
}
common26_orders = {name: common26 for name in datasets}
plot_fc_matrix_comparison(
    common26_mats,
    common26_orders,
    fig_out / "07_mean_fc_common26_comparison.png",
    "Mean functional connectivity comparison (shared TDBRAIN 26 channels)",
)

print(f"Figures saved to: {fig_out}")


Figures saved to: e:\Workspace\EEG_Preprocess\eeg_consistency_report_notebook\figures


In [32]:

# 列出输出文件
sorted([p.name for p in fig_out.glob("*.png")])


['01_window_rms_distribution_4panel.png',
 '02_mean_welch_psd.png',
 '03_mean_fft_spectrum_4panel.png',
 '04_relative_band_power_4panel.png',
 '05_alpha_topomap_comparison.png',
 '06_mean_fc_full_comparison.png',
 '07_mean_fc_common26_comparison.png']